In [1]:
import pandas as pd
from ast import literal_eval
import statsmodels.formula.api as smf

In [2]:
job_postings = pd.read_json("../data/job_postings.json")
job_postings.head()

,title,skills,years_experience,salary
0,Data Analyst,"[python, sql]",2,72000
1,Marketing Analyst,"[excel, sql]",1,65000
2,Business Analyst,"[excel, powerbi]",2,68000
3,Machine Learning Engineer,"[python, tensorflow, sql]",4,115000
4,Data Engineer,"[python, sql, aws]",3,105000


**Part 1:** Create a linear regression model with target salary using years_experience as a predictor.

In [3]:
salary_years_lm = smf.ols(formula='salary ~ years_experience', data=job_postings).fit()
salary_years_lm.params

Intercept           39542.753184
years_experience    18007.883566
dtype: float64

**Questions:** What does your model estimate for the average salary for jobs requiring 0 years of experience? 5 years of experience?

In [4]:
print(f"Averge salary for 0 years experience: ${salary_years_lm.params.Intercept.round(2)}")
print(f"Averge salary for 5 years experience: ${(salary_years_lm.params.Intercept + salary_years_lm.params.years_experience * 5).round(2)}")

Averge salary for 0 years experience: $39542.75
Averge salary for 5 years experience: $129582.17


**Part 2:** Now, let's see how different listed skills affect the salary.

First, [explode](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html) the skills column so that each different skill is listed on a different row. You should end up with 69 rows.

In [5]:
job_postings_explode_df = job_postings.explode(column="skills")
job_postings_explode_df.head(2)

,title,skills,years_experience,salary
0,Data Analyst,python,2,72000
0,Data Analyst,sql,2,72000


Next, let's create an [indicator variable](https://en.wikipedia.org/wiki/Dummy_variable_(statistics)) for each listed skill. We can do this using the [get_dummies function](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html). To make it easier on yourself later, use a prefix of "skill".

You should end up with a DataFrame with 69 rows and 7 columns.

In [6]:
job_postings_skills_df = pd.get_dummies(job_postings_explode_df, columns=['skills'], prefix=['skill'], dtype=int)
job_postings_skills_df.head()

,title,years_experience,salary,skill_aws,skill_excel,skill_powerbi,skill_python,skill_sql,skill_statistics,skill_tensorflow
0,Data Analyst,2,72000,0,0,0,1,0,0,0
0,Data Analyst,2,72000,0,0,0,0,1,0,0
1,Marketing Analyst,1,65000,0,1,0,0,0,0,0
1,Marketing Analyst,1,65000,0,0,0,0,1,0,0
2,Business Analyst,2,68000,0,1,0,0,0,0,0


Since we exploded the DataFrame earlier, we have multiple rows per listing. Use groupby (grouping on the index) and max to get one row per listing. You should end up with a DataFrame with 30 rows and 7 columns

In [7]:
job_postings_skills_df = job_postings_skills_df.groupby(level=0).max()
job_postings_skills_df.head(2)

,title,years_experience,salary,skill_aws,skill_excel,skill_powerbi,skill_python,skill_sql,skill_statistics,skill_tensorflow
0,Data Analyst,2,72000,0,0,0,1,1,0,0
1,Marketing Analyst,1,65000,0,1,0,0,1,0,0


In [8]:
job_postings.head(2)

,title,skills,years_experience,salary
0,Data Analyst,"[python, sql]",2,72000
1,Marketing Analyst,"[excel, sql]",1,65000


Finally, merge this back to the job_posting DataFrame. Hint: You can use the [join method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) to accomplish this.

In [9]:
job_postings_df = pd.merge(left=job_postings,
                           right=job_postings_skills_df,
                           on=["title", "years_experience", "salary"],
                           validate="1:1")
job_postings_df.head(2)

,title,skills,years_experience,salary,skill_aws,skill_excel,skill_powerbi,skill_python,skill_sql,skill_statistics,skill_tensorflow
0,Data Analyst,"[python, sql]",2,72000,0,0,0,1,1,0,0
1,Marketing Analyst,"[excel, sql]",1,65000,0,1,0,0,1,0,0


**Part 3:** Fit a linear regression model for salary using years_experience and each of the skill columns that you just created.

In [10]:
salary_years_lm = smf.ols(formula='salary ~ years_experience + skill_aws + skill_excel + skill_powerbi + skill_python + skill_sql + skill_statistics + skill_tensorflow', data=job_postings_skills_df).fit()
salary_years_lm.params

Intercept           44599.471401
years_experience    12724.399934
skill_aws           11018.047536
skill_excel           237.303547
skill_powerbi         248.857740
skill_python         7872.169132
skill_sql            2946.913611
skill_statistics     2916.569806
skill_tensorflow    12357.356923
dtype: float64

**Question:** Which skill increases the estimate for average salary the most?

skill_tensorflow    12357.356923